<a href="https://colab.research.google.com/github/KaveendraY/Regular-Db-vs-Vector-Search/blob/main/Regular_DB_vs_Vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn chromadb nest-asyncio requests -q

In [ ]:
from fastapi import FastAPI
import sqlite3
import chromadb

app = FastAPI(title="Database Comparison: Traditional vs Vector")

# 1. The Dataset: Unstructured tenant chat transcripts
transcripts = [
    {"id": "1", "text": "The heating in apartment 4B is broken and it is freezing."},
    {"id": "2", "text": "I need to renew my lease for another year."},
    {"id": "3", "text": "Can I pay my rent using a credit card?"},
    {"id": "4", "text": "The lock on the main courtyard gate is jammed."}
]

# 2. Traditional DB Setup (SQLite)
sql_conn = sqlite3.connect(":memory:", check_same_thread=False)
cursor = sql_conn.cursor()
cursor.execute("CREATE TABLE transcripts (id TEXT, text TEXT)")
for t in transcripts:
    cursor.execute("INSERT INTO transcripts VALUES (?, ?)", (t["id"], t["text"]))
sql_conn.commit()

def search_traditional(query: str):
    cursor.execute("SELECT id, text FROM transcripts WHERE text LIKE ?", (f"%{query}%",))
    return [{"id": row[0], "text": row[1]} for row in cursor.fetchall()]

# 3. Vector DB Setup (ChromaDB)
chroma_client = chromadb.Client()
# We use get_or_create_collection so you can re-run this cell without throwing an error
collection = chroma_client.get_or_create_collection(name="transcripts_collection")

# Only add documents if the collection is empty
if collection.count() == 0:
    collection.add(
        documents=[t["text"] for t in transcripts],
        ids=[t["id"] for t in transcripts]
    )

def search_vector(query: str):
    results = collection.query(query_texts=[query], n_results=1)
    return results

# 4. The Comparison Endpoint
@app.get("/search/compare")
def compare_search(query: str):
    trad_results = search_traditional(query)
    vec_results = search_vector(query)

    # Safely handle ChromaDB distance scores
    distance = vec_results["distances"][0][0] if vec_results["distances"] and vec_results["distances"][0] else 999

    return {
        "query": query,
        "traditional_database": {
            "success_rate": "100%" if len(trad_results) > 0 else "0%",
            "matches_found": len(trad_results),
            "results": trad_results
        },
        "vector_database": {
            "success_rate": "High Confidence" if distance < 1.5 else "Low Confidence",
            "matches_found": len(vec_results["ids"][0]),
            "results": vec_results["documents"][0],
            "distance_score": distance
        }
    }

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time

# Apply nest_asyncio to allow Uvicorn to run inside Jupyter's existing event loop
nest_asyncio.apply()

def run_app():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

# Start the server in a separate thread
server_thread = threading.Thread(target=run_app)
server_thread.start()

# Give the server a second to spin up
time.sleep(2)
print("✅ FastAPI Server is running in the background on http://127.0.0.1:8000")

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


✅ FastAPI Server is running in the background on http://127.0.0.1:8000


In [ ]:
import requests
import json

# Scenario 1: The user types an exact keyword from the database
response = requests.get("http://127.0.0.1:8000/search/compare?query=broken")
data = response.json()

print("SCENARIO 1: Exact Keyword ('broken')\n")
print(json.dumps(data, indent=2))

SCENARIO 1: Exact Keyword ('broken')

{
  "query": "broken",
  "traditional_database": {
    "success_rate": "100%",
    "matches_found": 1,
    "results": [
      {
        "id": "1",
        "text": "The heating in apartment 4B is broken and it is freezing."
      }
    ]
  },
  "vector_database": {
    "success_rate": "High Confidence",
    "matches_found": 1,
    "results": [
      "The heating in apartment 4B is broken and it is freezing."
    ],
    "distance_score": 1.4529975652694702
  }
}


In [ ]:
# Scenario 2: The user types a concept with NO matching keywords
response = requests.get("http://127.0.0.1:8000/search/compare?query=HVAC issues")
data = response.json()

print("SCENARIO 2: Semantic Concept ('HVAC issues')\n")
print(json.dumps(data, indent=2))

SCENARIO 2: Semantic Concept ('HVAC issues')

{
  "query": "HVAC issues",
  "traditional_database": {
    "success_rate": "0%",
    "matches_found": 0,
    "results": []
  },
  "vector_database": {
    "success_rate": "High Confidence",
    "matches_found": 1,
    "results": [
      "The heating in apartment 4B is broken and it is freezing."
    ],
    "distance_score": 1.191139578819275
  }
}
